# Bollworm Counting — full GPU run (Google Colab)

**Runtime → Change runtime type → T4 GPU** before running.

Before running, put these in your Google Drive (Drive stays private, which Zindi's rules require):
- `MyDrive/bollworm-counting-ai/` — this repo's code (no `data/` folder needed)
- `MyDrive/bollworm-data/` — the Zindi downloads (`Train.csv`, `Test.csv`, `SampleSubmission.csv`, `images_bboxes.csv`, `images.zip`)

Expected run time on a T4: data prep ~15–25 min, training (yolov8s, 1024px, 40 epochs) ~5–7 h, prediction ~15 min.
Colab may disconnect on long runs: weights are saved to Drive every epoch, and you can resume (cell 5).

In [ ]:
# 1. Mount Drive and copy code + data to fast local disk
from google.colab import drive
drive.mount('/content/drive')
!rsync -a --exclude data --exclude runs --exclude outputs /content/drive/MyDrive/bollworm-counting-ai/ /content/bollworm-counting-ai/
!mkdir -p /content/bollworm-counting-ai/data/raw && rsync -a --info=progress2 /content/drive/MyDrive/bollworm-data/ /content/bollworm-counting-ai/data/raw/
%cd /content/bollworm-counting-ai
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install (Colab already has CUDA torch)
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q

In [ ]:
# 3. Prepare data (unzip, fix orientation, resize, YOLO labels, train/val split)
!python -m bollworm.prepare_data --config configs/default.yaml
!python -m bollworm.visualize --config configs/default.yaml --n 6
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('outputs/viz/*.jpg'))[:3]:
    display(Image(p, width=700))  # boxes MUST sit on the moths -- if not, stop and investigate

In [ ]:
# 4. Baseline submission (median counts) -- upload it first to confirm the format is accepted
!python -m bollworm.baseline --config configs/default.yaml

In [ ]:
# 5. Train. Runs go straight to Drive so a disconnect doesn't lose progress.
RUNS = '/content/drive/MyDrive/bollworm-runs'
!sed -i "s#^  runs: .*#  runs: {RUNS}#" configs/default.yaml
RESUME = False   # set True after a disconnect
!python -m bollworm.train --config configs/default.yaml {'--resume' if RESUME else ''}

In [ ]:
# 6. Predict, tune per-class thresholds on val, write submission; save everything to Drive
!python -m bollworm.predict --config configs/default.yaml
!mkdir -p /content/drive/MyDrive/bollworm-outputs && cp -r outputs/* /content/drive/MyDrive/bollworm-outputs/
!ls -la /content/drive/MyDrive/bollworm-outputs